## Structured Output

Models can be requested to provide their response in a format matching a given schema. This can be done using Structured Output

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(model = "qwen/qwen3-32b")

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="Name of the movie")
    year:str=Field(description="The year in which movie is released")
    director:str=Field(description="Director of the movie")
    rating:str=Field(description="ration of movie out of 10")

In [7]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11ac48410>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10c6a03e0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'Name of the movie', 'type': 'string'}, 'year': {'description': 'The year in which movie is released', 'type': 'string'}, 'director': {'description': 'Director of the movie', 'type': 'string'}, 'rating': {'description': 'ration of movie out of 10', 'type': 'strin

In [8]:
response=model_with_structure.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year='2010', director='Christopher Nolan', rating='8.8')

### Message output alongside parsed structure

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(..., description="Name of the movie")
    year:str=Field(..., description="The year in which movie is released")
    director:str=Field(..., description="Director of the movie")
    rating:str=Field(..., description="ration of movie out of 10")

In [12]:
model_with_structure=model.with_structured_output(Movie, include_raw=True) 
model_with_structure.invoke("Give me details of movie yeh jawani hai deewani")

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Yeh Jawani Hai Deewani." Let me start by recalling what I know about this movie. It\'s a popular Indian film, so I need to make sure I get the details right. The function provided requires the title, year, director, and rating.\n\nFirst, the title is clearly "Yeh Jawani Hai Deewani." The year it was released—I think it came out in 2013. Let me confirm that. Yes, I\'m pretty sure it\'s 2013. The director is Rajkumar Hirani. Wait, no, actually, now that I think about it, Rajkumar Hirani directed "3 Idiots" and "PK," but "Yeh Jawani Hai Deewani" was directed by Ayan Mukerji. Oops, I almost made a mistake there. Better double-check that. Yes, Ayan Mukerji is the correct director. \n\nAs for the rating, the movie received positive reviews. On IMDb, it\'s around 8.2/10. Let me verify that. A quick mental check: yes, that\'s correct. So putting it all together, the para

In [14]:
# Nested Structure

from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class Movie(BaseModel):
    title:str
    year:str
    cast:list[Actor]
    genre:list[str]
    budget: float | None = Field(..., description="Budget in million USD")

In [15]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure.invoke("Provide me details of yeh jawani hai deewani")

Movie(title='Yeh Jawani Hai Deewani', year='2013', cast=[Actor(name='Ranbir Kapoor', role='Bobby Singh Ralakhan'), Actor(name='Deepika Padukone', role='Dimple Chaudhary'), Actor(name='Aditya Roy Kapur', role='Raju Singh Ralakhan')], genre=['Romantic Comedy', 'Coming-of-age'], budget=15.0)

# TypeDict
TypedDict is a special typing feature in Python used to define the structure of dictionaries. It does not do runtime validation like Pydantic.

In [17]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year in which the movie is released"]
    director: Annotated[str, ..., "Director of the movie"]
    rating: Annotated[float, ..., "rating of the movie out of 10"]

model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("Provide details of titanic movie")
response

{'director': 'James Cameron', 'rating': 7.8, 'title': 'Titanic', 'year': 1997}

In [19]:
class Actor(TypedDict):
    name : str
    role : str

class Movie(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genre: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_typedict = model.with_structured_output(Movie)
response = model_with_typedict.invoke("Provide details of titanic movie")
response

{'budget': 200000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Jack'},
  {'name': 'Kate Winslet', 'role': 'Rose'}],
 'genre': ['Romance', 'Drama'],
 'title': 'Titanic',
 'year': 1997}